# Label Distributions

How the adjudicator's labels are distributed, and how they vary by topic.
Setting and agency are both 1–5 Likert; event relation is categorical.

Topics with fewer than `MIN_TOPIC_N` instances are dropped from the
rating-by-topic heatmaps, since a mean over three instances is noise.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import seaborn as sns

from nb_utils import (
    setup_plots, load, annotator_labels, short_name,
    DIMENSIONS, TASK_COLOR, SCALE_15,
)

plt = setup_plots()

ANNOTATOR    = 'tejo9855'
MIN_TOPIC_N  = 5      # fewest instances a topic needs to appear in the heatmaps
SCALE_MIN, SCALE_MAX = SCALE_15[0], SCALE_15[-1]   # 1-5 for both Likert tasks

In [ ]:
setting_df = annotator_labels('setting', ANNOTATOR)
agency_df  = annotator_labels('agency',  ANNOTATOR)

meta = (load('corpus')[['safe_instance_id', 'topic_classification', 'folder',
                        'dolma_source', 'is_noise', 'narrative_label']]
        .rename(columns={'dolma_source': 'source'}))

# 'Unknown' topics carry no signal on their own, so they are relabelled by the
# shard they came from -- every instance then has a meaningful group.
unknown = meta['topic_classification'] == 'Unknown'
meta.loc[unknown, 'topic_classification'] = 'Shard: ' + meta.loc[unknown, 'source'].str.title()

setting_df = setting_df.merge(meta, on='safe_instance_id', how='left')
agency_df  = agency_df.merge(meta,  on='safe_instance_id', how='left')

SETTING_DIMS = DIMENSIONS['setting']
AGENCY_DIMS  = DIMENSIONS['agency']
SHORT = {d: short_name(d) for d in SETTING_DIMS + AGENCY_DIMS}

both = set(setting_df.safe_instance_id) & set(agency_df.safe_instance_id)
print(f'Setting: {len(setting_df)}   Agency: {len(agency_df)}   Both tasks: {len(both)}')

## 1. Topic distribution of annotated instances

In [ ]:
topic_counts = pd.concat([
    setting_df['topic_classification'].value_counts().rename('Setting'),
    agency_df['topic_classification'].value_counts().rename('Agency'),
], axis=1).fillna(0).astype(int)

eligible = (topic_counts[topic_counts.max(axis=1) >= MIN_TOPIC_N]
            .sort_values('Setting'))

fig, ax = plt.subplots(figsize=(9, 7))
eligible.plot(kind='barh', ax=ax, edgecolor='white',
              color=[TASK_COLOR['setting'], TASK_COLOR['agency']])
ax.axvline(MIN_TOPIC_N, color='#c00', linestyle='--', linewidth=0.9, alpha=0.7,
           label=f'min = {MIN_TOPIC_N}')
ax.set_title(f'Annotated instances per topic — {ANNOTATOR} '
             f'(topics with ≥ {MIN_TOPIC_N} in either task)')
ax.set_xlabel('Instances')
ax.set_ylabel('')
ax.legend()
plt.tight_layout()
plt.show()

print(eligible.sort_values('Setting', ascending=False).to_string())

## 2. Ratings by topic

Mean rating per dimension, for topics clearing `MIN_TOPIC_N`. Both heatmaps are
scaled 1–5, the full Likert range, so the two tasks are directly comparable.

In [ ]:
def ratings_by_topic(df, dims, cmap, title):
    keep = df['topic_classification'].value_counts()
    keep = keep[keep >= MIN_TOPIC_N].index
    by_topic = (df[df['topic_classification'].isin(keep)]
                .groupby('topic_classification')[dims].mean()
                .rename(columns=SHORT)
                .sort_index())

    fig, ax = plt.subplots(figsize=(len(dims) * 2 + 2, max(5, len(by_topic) * 0.42)))
    sns.heatmap(by_topic, annot=True, fmt='.2f', cmap=cmap,
                vmin=SCALE_MIN, vmax=SCALE_MAX, linewidths=0.3, ax=ax,
                annot_kws={'size': 9})
    ax.set_title(f'{title} — {ANNOTATOR} (topics with ≥ {MIN_TOPIC_N} instances)')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
    return by_topic


_ = ratings_by_topic(setting_df, SETTING_DIMS, 'YlGn', 'Mean setting ratings by topic')
_ = ratings_by_topic(agency_df,  AGENCY_DIMS,  'YlOrRd', 'Mean agency ratings by topic')

## 3. Label distributions

In [ ]:
n_cols = max(len(SETTING_DIMS), len(AGENCY_DIMS))
fig, axes = plt.subplots(2, n_cols, figsize=(3.4 * n_cols, 7))

for row, (dims, task) in enumerate([(SETTING_DIMS, 'setting'), (AGENCY_DIMS, 'agency')]):
    for col_i, (ax, dim) in enumerate(zip(axes[row], dims)):
        counts = (setting_df if task == 'setting' else agency_df)[dim] \
                 .value_counts().reindex(SCALE_15, fill_value=0)
        ax.bar(counts.index, counts.to_numpy(), color=TASK_COLOR[task],
               edgecolor='white', width=0.6)
        ax.set_title(SHORT[dim], fontsize=10)
        ax.set_xlabel(f'Rating ({SCALE_MIN}–{SCALE_MAX})')
        ax.set_xticks(SCALE_15)
        # Only the leftmost panel is labelled -- the row name sits there too, and
        # repeating 'Count' five times crowds it.
        ax.set_ylabel('Count' if col_i else '')
    # Hide leftover axes in whichever row has fewer dimensions.
    for ax in axes[row][len(dims):]:
        ax.set_visible(False)

for row, label in enumerate(['Setting', 'Agency']):
    axes[row][0].annotate(label, xy=(-0.42, 0.5), xycoords='axes fraction',
                          fontsize=12, fontweight='bold', va='center', rotation=90)

plt.suptitle(f'Label distributions — {ANNOTATOR}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
event_df = load('event_relation')
event_cols = [f'{d}_{ANNOTATOR}' for d in DIMENSIONS['event_relation']
              if f'{d}_{ANNOTATOR}' in event_df.columns]

fig, axes = plt.subplots(1, len(event_cols), figsize=(3.8 * len(event_cols), 4))
for ax, col in zip(axes, event_cols):
    counts = event_df[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.to_numpy(),
           color=TASK_COLOR['event_relation'], edgecolor='white', width=0.6)
    ax.set_title(short_name(col.replace(f'_{ANNOTATOR}', '')), fontsize=10)
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle(f'Event relation label distributions — {ANNOTATOR} (n={len(event_df)})',
             fontsize=13)
plt.tight_layout()
plt.show()

## 4. Cross-task correlation: setting × agency

Pearson r across the instances annotated in both tasks. Each pair is computed on
the instances where both dimensions have a value.

In [ ]:
ALL_DIMS = SETTING_DIMS + AGENCY_DIMS

cross_df = (setting_df[['safe_instance_id'] + SETTING_DIMS]
            .merge(agency_df[['safe_instance_id'] + AGENCY_DIMS],
                   on='safe_instance_id', how='inner'))
print(f'Instances annotated in both tasks: {len(cross_df)}')
print('Missing values per dimension:')
print(cross_df[ALL_DIMS].isna().sum().to_string())

corr_vals = pd.DataFrame(index=ALL_DIMS, columns=ALL_DIMS, dtype=float)
pval_vals = pd.DataFrame(index=ALL_DIMS, columns=ALL_DIMS, dtype=float)
for d1 in ALL_DIMS:
    for d2 in ALL_DIMS:
        ok = cross_df[d1].notna() & cross_df[d2].notna()
        r, p = pearsonr(cross_df.loc[ok, d1], cross_df.loc[ok, d2])
        corr_vals.loc[d1, d2], pval_vals.loc[d1, d2] = r, p

corr_display = corr_vals.rename(index=SHORT, columns=SHORT).astype(float)

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(corr_display, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.3, ax=ax,
            annot_kws={'size': 8},
            mask=np.triu(np.ones_like(corr_display, dtype=bool), k=1))
ax.axhline(len(SETTING_DIMS), color='black', linewidth=2)
ax.axvline(len(SETTING_DIMS), color='black', linewidth=2)
ax.set_title(f'Pearson r — setting × agency ({ANNOTATOR}, n={len(cross_df)})')
plt.tight_layout()
plt.show()

In [ ]:
cross_corrs = [
    {'setting': SHORT[sd], 'agency': SHORT[ad],
     'r': corr_vals.loc[sd, ad], 'p': pval_vals.loc[sd, ad]}
    for sd in SETTING_DIMS for ad in AGENCY_DIMS
]
cross_corr_df = (pd.DataFrame(cross_corrs)
                 .sort_values('r', key=abs, ascending=False)
                 .reset_index(drop=True)
                 .round({'r': 3, 'p': 4}))
print('Strongest cross-task correlations:')
print(cross_corr_df.head(10).to_string(index=False))

In [ ]:
# Scatter of the six strongest cross-task pairs. Ratings are discrete, so points
# are jittered -- otherwise every instance lands on one of 25 spots.
rng = np.random.default_rng(42)
top6 = cross_corr_df.head(6)
inv = {v: k for k, v in SHORT.items()}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (_, row) in zip(axes.flat, top6.iterrows()):
    sd, ad = inv[row['setting']], inv[row['agency']]
    jitter = lambda s: s + rng.uniform(-0.15, 0.15, len(s))
    ax.scatter(jitter(cross_df[sd]), jitter(cross_df[ad]),
               alpha=0.4, s=25, color='#555', edgecolors='white', linewidths=0.3)
    ax.set_xlabel(f'setting: {row["setting"]}', fontsize=9)
    ax.set_ylabel(f'agency: {row["agency"]}', fontsize=9)
    ax.set_title(f'r = {row["r"]:.3f}  (p = {row["p"]:.3f})', fontsize=9)

plt.suptitle(f'Six strongest cross-task correlations — {ANNOTATOR}', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Export the correlation matrix for the paper

In [ ]:
out_path = f'../figures/pearson_feature_correlations_{ANNOTATOR}.tex'
latex = corr_display.round(2).to_latex(
    float_format='%.2f', na_rep='',
    caption=(f'Pearson correlations between setting and agency dimensions '
             f'({ANNOTATOR}, n={len(cross_df)}).'),
    label='tab:pearson_feature_corr', position='ht',
)
with open(out_path, 'w') as f:
    f.write(latex)
print(f'Wrote {out_path}')